# ARCHS4 model building with CLAMP

💡 **Environment:** `clamp-analyses`  

This notebook focuses on preprocessing data from the ARCHS4 database. It covers initial cleaning, and preparation for subsequent analyses.

## Load libraries

In [ ]:
library(hdf5r)
library(biomaRt)
library(dplyr)
library(here)
library(CLAMP)

source(here("config.R"))
set.seed(config$ARCHS4$RANDOM_SVD_SEED)


Attaching package: ‘dplyr’


The following object is masked from ‘package:biomaRt’:

    select


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


here() starts at /home/msubirana/Documents/pivlab/clamp-analyses



## Output directory

In [ ]:
output_dir <- config$ARCHS4$DATASET_FOLDER
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

## Preprocess archs4 data

In [ ]:
file_path <- config$ARCHS4$DATASET_FILE

h5        <- H5File$new(file_path, mode = "r")
dset      <- h5[["/data/expression"]]
gene_symbols <- h5[["/meta/genes/symbol"]]$read()
gene_ids     <- h5[["/meta/genes/ensembl_gene"]]$read()
sample_names <- h5[["/meta/samples/geo_accession"]]$read()
sc_samples      <- h5[["/meta/samples/singlecellprobability"]]$read()
lib_strategy <- h5[["/meta/samples/library_strategy"]]$read()     

Get genes length

In [ ]:
# # run it with internet connection

# output_file <- here('data/archs4/gene_lengths.rds')

# ensembl_ver <- config$ARCHS4$DATASET_ENSEMBL_VERSION
# ensembl <- useEnsembl(biomart="ensembl", version=ensembl_ver)
# mart <- biomaRt::useDataset("hsapiens_gene_ensembl", ensembl)

# gene_info <- biomaRt::getBM(
#   filters = "hgnc_symbol",
#   attributes = c("ensembl_gene_id", "hgnc_symbol", "transcript_length"),
#   values = gene_symbols, 
#   mart = mart
# )

# gene_lengths <- gene_info %>%
#   group_by(hgnc_symbol, ensembl_gene_id) %>%
#   summarize(gene_length_bp = max(transcript_length, na.rm = TRUE)) %>%
#   pull(gene_length_bp, name = hgnc_symbol)

# saveRDS(gene_lengths, file = output_file)

`summarise()` has grouped output by 'hgnc_symbol'. You can override using the
`.groups` argument.


In [ ]:
output_file <- here('data/archs4/gene_lengths.rds')
gene_lengths <- readRDS(output_file)

In [ ]:
# keep in data only genes for which we have lengths
# here gene symbols are duplicated (we'll use that later)
gene_symbols_idx <- which( gene_symbols %in% names(gene_lengths) )
gene_symbols_thin <- gene_symbols[gene_symbols_idx]

# get unique gene symbols
# since we are gonna use rowsum to aggregate same gene symbols, I use it here
# to get a list of unique and fixed gene symbols
summed <- rowsum(t(dset[1:10, gene_symbols_idx]), group = gene_symbols_thin)
gene_symbols_thin_unique <- rownames(summed)
stopifnot(length(gene_symbols_thin_unique) == length(unique(gene_symbols_thin)))
gene_lengths <- gene_lengths[gene_symbols_thin_unique]
n_genes_thin <- length(gene_symbols_thin_unique)
rm(summed)

## FMB matrix

In [ ]:
fbm_file  <- file.path(output_dir, "fbm")

fbm_file_path <- paste0(fbm_file, ".bk")
if (file.exists(fbm_file_path)) {
  file.remove(fbm_file_path)
}

fbm_obj <- FBM(
  nrow        = n_genes_thin,
  ncol        = n_samples,
  backingfile = fbm_file,
  create_bk   = TRUE,
)

✅ Removed existing FBM file: /home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/FBMarchs4.bk

ℹ️ Not found, skipping: /home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/FBMarchs4_preproc.bk

ℹ️ Not found, skipping: /home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/FBMarchs4_filtered.bk

ℹ️ Not found, skipping: /home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/FBMarchs4_preproc_filtered.bk

ℹ️ Not found, skipping: /home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/FBMarchs4_preproc_filtered.rds

✅ Removed existing FBM file: /home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/FBMarchs4_nosinglecell.bk

✅ Removed existing FBM file: /home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/FBMarchs4_nosinglecell_preproc.bk

✅ Removed existing FBM file: /home/msubirana/Documents/pivlab/clamp-analyses/output/archs4/FBMarchs4_nosinglecell_preproc_filtered.bk

ℹ️ Not found, skipping: /home/msubirana/Documents/pivlab/clamp-a

In [ ]:
tpm_norm <- function(counts, gene.lengths) {
  if (!is.matrix(counts)) {
    stop("`counts` must be a matrix.")
  }
  counts <- as.matrix(counts)
  
  if (is.null(names(gene.lengths))) {
    stop("`gene.lengths` must be a named numeric vector.")
  }
  if (!all(colnames(counts) %in% names(gene.lengths))) {
    stop("All column names of `counts` must be in names(gene.lengths).")
  }
  
  # Reorder lengths to match columns
  #lengths.bp <- gene.lengths[colnames(counts)]
  
  # Convert lengths to kilobases
  lengths.kb <- gene.lengths / 1e3
  
  # 1) Divide counts by gene length in kilobases → reads per kilobase
  rpk <- sweep(counts, 2, lengths.kb, FUN = "/")
  
  # 2) Compute per-sample scaling factor: sum of RPKs
  per.sample.sum <- rowSums(rpk)
  
  # 3) Divide RPKs by the sum and multiply by 1e6 → TPM
  tpm <- sweep(rpk, 1, per.sample.sum, FUN = "/") * 1e6
  
  return(tpm)
}

In [ ]:
# FIXME: here I override the CHUNK_SIZE value since there is an error in reading
# ARCHS4 at some block
block_size <- 100 # config$GENERAL$CHUNK_SIZE
n_blocks   <- ceiling(n_samples / block_size)

pb <- txtProgressBar(min = 0, max = n_blocks, style = 3)

for (i in 1:n_blocks) {
  setTxtProgressBar(pb, i)

  start_row <- (i-1) * block_size + 1
  end_row <- min(i * block_size, n_samples)

  raw_block <- NULL
  tryCatch(
    expr = {
      raw_block <- dset[start_row:end_row, ]
    },
    error = function(e) {
      message(paste("Error with block: ", i))
    }
  )
  if (is.null(raw_block)) {
    # FIXME: I add a dummy block here with near-to-zero values; not sure if
    # this is the best approach here; for some reason, there is a block that
    # cannot be read in ARCHS4, so I "skip" it here
    raw_block <- matrix(1e-10, (end_row - start_row + 1), n_genes)
  }

  # subset genes to those with gene length only
  raw_block <- raw_block[, gene_symbols_idx]

  # aggregate duplicated gene symbols
  raw_block_t <- t(raw_block)
  raw_block_summed <- rowsum(raw_block_t, group = gene_symbols_thin)
  raw_block <- t(raw_block_summed)
  rownames(raw_block) <- NULL

  # tpm normalize
  raw_block <- t(tpm_norm(raw_block, gene_lengths))

  fbm_obj[, start_row:end_row] <- as.matrix(raw_block)
}

## Clean FMB matrix

In [ ]:
N_CORES <- config$GENERAL$N_CORES
if (N_CORES > 1) {
  # if we are parallelizing, then disable BLAS parallelization
  options(bigstatsr.check.parallel.blas = FALSE)
  blas_nproc <- getOption("default.nproc.blas")
  options(default.nproc.blas = NULL)
}

# run cleanFBM (log, NAs handling, etc)
(fbm_obj, ncores=N_CORES)

if (N_CORES > 1) {
  options(bigstatsr.check.parallel.blas = TRUE)
  options(default.nproc.blas = blas_nproc)
}

## Row stats

In [ ]:
N_CORES <- config$GENERAL$N_CORES
if (N_CORES > 1) {
  # if we are parallelizing, then disable BLAS parallelization
  options(bigstatsr.check.parallel.blas = FALSE)
  blas_nproc <- getOption("default.nproc.blas")
  options(default.nproc.blas = NULL)
}

rowStats=computeRowStatsFBM(fbm_obj, ncores=N_CORES)

if (N_CORES > 1) {
  options(bigstatsr.check.parallel.blas = TRUE)
  options(default.nproc.blas = blas_nproc)
}

## Filtering

In [ ]:
samples_idx <- which(sc_samples < 0.5)
n_samples <- length(samples_idx)

filterResult=filterFBM(
	fbm_obj,
	rowStats,
  keep_samples_idx = samples_idx,
	mean_cutoff = config$ARCHS4$GENES_MEAN_CUTOFF,
	var_cutoff = config$ARCHS4$GENES_VAR_CUTOFF,
	backingfile = output_file
)

fbm_obj_filtered=filterResult$fbm_filtered

In [ ]:
message(paste("New filtered dataset dims: ", nrow(fbm_obj_filtered), " x ", ncol(fbm_obj_filtered)))

In [ ]:
gene_symbols_thin=gene_symbols_thin[filterResult$kept_rows]
n_genes_thin <- length(gene_symbols_thin)

In [ ]:
saveRDS(
  list(
#    gene_symbols_idx=gene_symbols_idx,
    gene_symbols_thin=gene_symbols_thin,
    gene_lengths=gene_lengths,
    n_genes_thin=n_genes_thin,
    n_samples=n_samples
  ),
  file.path(output_dir, "metadata_filtered.rds")
)

In [ ]:
message(paste("Number of genes kept: ", length(gene_symbols_thin)))
message(paste("Number of samples kept: ", n_samples))

In [ ]:
saveRDS(sample_names, file.path(output_dir, "all_samples.rds"))

In [ ]:
rowStats$row_means=rowStats$row_means[filterResult$kept_rows]
rowStats$row_variances=rowStats$row_variances[filterResult$kept_rows]

## Zscore

In [ ]:
zscoreFBM(fbm_obj_filtered, rowStats = rowStats, chunk_size=config$GENERAL$CHUNK_SIZE)

In [ ]:
h5$close_all()